# VisoMaster Setup Notebook

This notebook handles the setup procedures for VisoMaster:
1. Installing required dependencies
2. Fixing Models.py backslash path issue
3. Downloading models

Run the cells in order to set up the environment. All operations are logged for debugging.

In [ ]:
# Setup logging system
import logging
import os
import sys
import datetime
import traceback

# Create logs directory if it doesn't exist
logs_dir = os.path.join(os.getcwd(), 'Logs')
os.makedirs(logs_dir, exist_ok=True)

# Configure logging
log_filename = os.path.join(logs_dir, f'setup_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger('visomaster_setup')

logger.info("=== VisoMaster Setup Started ===")
logger.info(f"Log file created at: {log_filename}")
logger.info(f"Current working directory: {os.getcwd()}")

In [ ]:
# Error handling decorator
def log_step(func):
    def wrapper(*args, **kwargs):
        step_name = func.__name__
        logger.info(f"Starting step: {step_name}")
        try:
            result = func(*args, **kwargs)
            logger.info(f"Successfully completed step: {step_name}")
            return result
        except Exception as e:
            logger.error(f"Error in step {step_name}: {str(e)}")
            logger.error(traceback.format_exc())
            raise
    return wrapper

In [ ]:
# Install scikit-image with conda
@log_step
def install_scikit_image():
    import subprocess
    logger.info("Installing scikit-image with conda...")
    result = subprocess.run(['conda', 'install', '-y', 'scikit-image'], 
                           capture_output=True, text=True)
    
    if result.returncode != 0:
        logger.error(f"Failed to install scikit-image: {result.stderr}")
        raise RuntimeError("Failed to install scikit-image")
        
    logger.info(result.stdout)
    return "✅ scikit-image installed"

print(install_scikit_image())

In [ ]:
# Install dependencies from requirements_cu124.txt
@log_step
def install_requirements():
    import subprocess
    # First check if we're in the VisoMaster directory
    current_dir = os.getcwd()
    logger.info(f"Current directory: {current_dir}")
    
    if 'visomaster' in current_dir.lower():
        visomaster_dir = current_dir
    else:
        visomaster_dir = '/workspace/visomaster'
        logger.info(f"Changing directory to: {visomaster_dir}")
        os.chdir(visomaster_dir)
    
    # Check if requirements file exists
    req_file = os.path.join(visomaster_dir, 'requirements_cu124.txt')
    if not os.path.exists(req_file):
        logger.error(f"Requirements file not found: {req_file}")
        raise FileNotFoundError(f"Requirements file not found: {req_file}")
    
    logger.info(f"Installing dependencies from: {req_file}")
    result = subprocess.run(['pip', 'install', '-r', req_file, '--no-cache-dir'],
                           capture_output=True, text=True)
    
    if result.returncode != 0:
        logger.error(f"Failed to install requirements: {result.stderr}")
        raise RuntimeError("Failed to install requirements")
        
    logger.info("Requirements installation completed")
    return f"✅ Dependencies from {req_file} installed"

print(install_requirements())

In [ ]:
# Download models
@log_step
def download_models():
    import subprocess
    # Navigate to model_assets directory
    current_dir = os.getcwd()
    if 'visomaster' in current_dir.lower():
        visomaster_dir = current_dir
    else:
        visomaster_dir = '/workspace/visomaster'
    
    model_assets_dir = os.path.join(visomaster_dir, 'model_assets')
    logger.info(f"Changing directory to: {model_assets_dir}")
    
    if not os.path.exists(model_assets_dir):
        logger.error(f"Model assets directory not found: {model_assets_dir}")
        raise FileNotFoundError(f"Model assets directory not found: {model_assets_dir}")
    
    os.chdir(model_assets_dir)
    
    # Check if download_models.py exists
    if not os.path.exists('download_models.py'):
        logger.error("download_models.py script not found")
        raise FileNotFoundError("download_models.py script not found")
    
    logger.info("Running download_models.py script...")
    result = subprocess.run(['python', 'download_models.py'],
                           capture_output=True, text=True)
    
    if result.returncode != 0:
        logger.error(f"Failed to download models: {result.stderr}")
        raise RuntimeError("Failed to download models")
    
    logger.info(result.stdout)
    logger.info("Models download completed")
    return "✅ Models downloaded"

print(download_models())

In [ ]:
# Verify installation
@log_step
def verify_installation():
    # Determine paths
    current_dir = os.getcwd()
    if 'model_assets' in current_dir.lower():
        model_dir = current_dir
        visomaster_dir = os.path.dirname(current_dir)
    elif 'visomaster' in current_dir.lower():
        visomaster_dir = current_dir
        model_dir = os.path.join(visomaster_dir, 'model_assets')
    else:
        visomaster_dir = '/workspace/visomaster'
        model_dir = os.path.join(visomaster_dir, 'model_assets')
    
    # Check if model directory exists
    if not os.path.exists(model_dir):
        logger.error(f"Model directory not found: {model_dir}")
        raise FileNotFoundError(f"Model directory not found: {model_dir}")
    
    # Check if model files exist
    model_files = os.listdir(model_dir)
    logger.info(f"Found {len(model_files)} files in model directory")
    
    for file in model_files:
        file_path = os.path.join(model_dir, file)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
        logger.info(f"Model file: {file}, Size: {file_size:.2f} MB")
    
    if len(model_files) == 0:
        logger.warning("No model files found!")
    
    logger.info("Installation verification completed")
    return "\n✅ Setup complete! VisoMaster is ready to use."

print(verify_installation())